In [27]:
import numpy as np
import pandas

import matplotlib.pyplot as plt
from tqdm.notebook import tqdm

SEED = 123456789
rng = np.random.default_rng(SEED)

In [6]:
data = pandas.read_csv('../data/ml-latest-small/ratings.csv')

In [7]:
data

,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931
...,...,...,...,...
100831,610,166534,4.0,1493848402
100832,610,168248,5.0,1493850091
100833,610,168250,5.0,1494273047
100834,610,168252,5.0,1493846352


In [14]:
user_mapping = {}
movie_mapping = {}

user_counter = 0
movie_counter = 0

for k in range(len(data)):
    row = data.iloc[k]
    userId = int(row['userId'])
    movieId = int(row['movieId'])

    if userId not in user_mapping:
        user_mapping[userId] = user_counter
        user_counter += 1

    if movieId not in movie_mapping:
        movie_mapping[movieId] = movie_counter
        movie_counter += 1

In [16]:
n_u = len(user_mapping)
n_i = len(movie_mapping)

print(n_u, n_i)

610 9724


In [28]:
k = 8
### users
P = rng.normal(size=(n_u, k)) / np.sqrt(k)
### items
Q = rng.normal(size=(n_i, k)) / np.sqrt(k)

In [29]:
### predictions for user 7 and item 5
P[7] @ Q[5] + mean_R

np.float64(3.425797572141131)

In [31]:
#### user -> list of items + ratings
user_index = {}
### item -> list of users + ratings
item_index = {}

for k in range(len(data)):
    row = data.iloc[k]
    userId = int(row['userId'])
    movieId = int(row['movieId'])
    
    user_i = user_mapping[userId]
    item_i = movie_mapping[movieId]
    
    rating = float(row['rating'])

    if user_i not in user_index:
        user_index[user_i] = list()
    user_index[user_i].append((item_i, rating))

    if item_i not in item_index:
        item_index[item_i] = list()
    item_index[item_i].append((user_i, rating))

In [39]:
def linear_regression(X, y, alpha=1.0e-3):
    n_f = X.shape[1]

    A = X.T @ X + alpha * np.eye(n_f)
    b = X.T @ y

    return np.linalg.solve(A, b)

In [44]:
alpha = 1.0e-2
n_iterations = 100

for iteration in tqdm(range(n_iterations)):
    for i in range(n_u):
        items, ratings = zip(*user_index[i])
        Q_u = Q[items, :]
        r_u = np.array(ratings) - mean_R
        P[i] = linear_regression(Q_u, r_u, alpha=alpha)

    for j in range(n_i):
        users, ratings = zip(*item_index[j])
        P_i = P[users, :]
        r_i = np.array(ratings) - mean_R
        Q[j] = linear_regression(P_i, r_i, alpha=alpha)

  0%|          | 0/100 [00:00<?, ?it/s]